In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
class LEVIRCDDataset(Dataset):
    def __init__(self, images_dir, transform=None):
        # caminhos para imagens pré-mudança (A), pós-mudança (B) e máscaras
        self.images_dir_a = os.path.join(images_dir, 'A')  # antes
        self.images_dir_b = os.path.join(images_dir, 'B')  # depois
        self.masks_dir = os.path.join(images_dir, 'label') # máscara
        self.transform = transform

        # lista ordenada de arquivos de imagem em cada diretório
        self.images_a = sorted(f for f in os.listdir(self.images_dir_a) 
                               if f.endswith(('.png', '.jpg', '.jpeg')))
        self.images_b = sorted(f for f in os.listdir(self.images_dir_b) 
                               if f.endswith(('.png', '.jpg', '.jpeg')))
        self.masks = sorted(f for f in os.listdir(self.masks_dir) 
                            if f.endswith(('.png', '.jpg', '.jpeg')))

    def __len__(self):
        return len(self.images_a)

    def __getitem__(self, idx):
        # monta caminhos completos para o par de imagens e a máscara correspondente
        img_path_a = os.path.join(self.images_dir_a, self.images_a[idx])
        img_path_b = os.path.join(self.images_dir_b, self.images_b[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])
                
        # leitura e pré-processamento das imagens: BGR → RGB → float normalizado [0, 1]
        image_a = cv2.imread(img_path_a, cv2.IMREAD_COLOR)
        image_a = cv2.cvtColor(image_a, cv2.COLOR_BGR2RGB)
        image_a = image_a.astype(np.float32) / 255.0

        image_b = cv2.imread(img_path_b, cv2.IMREAD_COLOR)
        image_b = cv2.cvtColor(image_b, cv2.COLOR_BGR2RGB)
        image_b = image_b.astype(np.float32) / 255.0

        # leitura da máscara binária em escala de cinza e normalização para [0, 1]        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = mask.astype(np.float32) / 255.0

        # aplica augmentações sincronizadas nas duas imagens e na máscara
        if self.transform:
            augmented = self.transform(image=image_a, image1=image_b, mask=mask)
            image_a = augmented['image']
            image_b = augmented['image1']
            mask = augmented['mask']
                
        # adiciona dimensão de canal à máscara: (H, W) → (1, H, W)
        mask = mask.unsqueeze(0)

        return image_a, image_b, mask

# transformações albumentations
# treino: redimensionamento + augmentações geométricas aleatórias + conversão para tensor
train_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2(),
], additional_targets={'image1': 'image'})  # sincroniza augmentação entre imagem pré e pós

# validação: apenas redimensionamento e conversão para tensor (sem augmentação)
val_transform = A.Compose([
    A.Resize(height=256, width=256),
    ToTensorV2(),
], additional_targets={'image1': 'image'})


In [3]:
# caminho raiz do dataset LEVIR-CD e tamanho do batch
root = r"C:\\Users\\linky\\Downloads\\estudos_\\cards_testes\\projeto\\LEVIR_CD"
batch_size = 16

LEVIR_train_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'train'),
    transform = train_transform
)

LEVIR_val_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'val'),
    transform = val_transform
)

LEVIR_test_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'test'),
    transform = val_transform # sem augmentação, igual à validação
)

# criação dos DataLoaders para treino, validação e teste
# shuffle = True no treino para embaralhar os batches e melhorar a generalização
LEVIR_train_loader = DataLoader(LEVIR_train_dataset, batch_size=batch_size, shuffle=True)
LEVIR_val_loader = DataLoader(LEVIR_val_dataset, batch_size=batch_size, shuffle=False)
LEVIR_test_loader = DataLoader(LEVIR_test_dataset, batch_size=batch_size, shuffle=False)